# 04 — Fama–MacBeth & Performance Metrics

Characteristic cross-sectional regressions with **Newey–West** standard errors,
plus Sharpe ($\sqrt{252}$), max drawdown, and quantile diagnostics.

$$
R_{i,t}-R_{f,t}=\gamma_{0,t}+\gamma_{1,t}\,SMB_{i,t}+\gamma_{2,t}\,HML_{i,t}+\gamma_{3,t}\,WML_{i,t}+\gamma_{4,t}\,ILLIQ_{i,t}+\varepsilon_{i,t}
$$

In [ ]:
import polars as pl

from src.data.store import read_partitioned_parquet
from src.data.paths import PROCESSED_FACTORS_DIR, PROCESSED_RETURNS_DIR
from src.backtest import (
    BacktestConfig,
    run_backtest,
    run_quantile_spread,
    fama_macbeth_characteristics,
    summarize_returns,
    quantile_metrics,
    export_fm_report,
    export_backtest_report,
)

chars = read_partitioned_parquet(PROCESSED_FACTORS_DIR / "characteristics")
print("rows", chars.height)
chars.head()

## Fama–MacBeth premia (Newey–West t-stats)

In [ ]:
char_cols = [
    c for c in ("log_me_z", "btm_z", "mom_12_1_z", "illiq_signal_z",
                "log_me", "btm", "mom_12_1", "illiq_signal")
    if c in chars.columns
]
# Prefer z-scored winsorized signals when present
z_cols = [c for c in char_cols if c.endswith("_z")]
use_cols = z_cols[:4] if z_cols else char_cols[:4]
print("regressors:", use_cols)

if chars.height and use_cols:
    ret_col = "excess_ret" if "excess_ret" in chars.columns else "ret"
    fm = fama_macbeth_characteristics(
        chars, use_cols, ret_col=ret_col, nw_lags=None, min_obs=30
    )
    display(fm.lambdas)
    export_fm_report(fm, stem="nse_fm")
else:
    fm = None
    print("Build Phase 2–3 panels first.")

## ILLIQ long–short backtest metrics

In [ ]:
signal = "illiq_signal" if chars.height and "illiq_signal" in chars.columns else None
if signal:
    bt = run_backtest(chars, BacktestConfig(signal_col=signal, kind="quintile", long_high=True))
    print(summarize_returns(bt.returns, ret_col="net_ret"))
    spread = run_quantile_spread(chars, signal, kind="quintile")
    display(quantile_metrics(spread))
    export_backtest_report(
        bt.returns, weights=bt.weights, quantile_returns=spread, stem="illiq_ls"
    )
else:
    print("No illiq_signal column.")